# Random Forests and Gradient Boosting

Implement a Random Forest (bootstrap-bagging of shallow decision trees with Gini impurity and random feature subsets) and a gradient-boosted tree ensemble from scratch; validate both against scikit-learn equivalents on a binary classification dataset.

## Configuration

Device, seed, and dtype come from `config.toml` via `shared.config.configure()`.  scikit-learn and NumPy operations run on CPU; the `device` variable is kept for consistency with other notebooks.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)


running on: mps


## Dataset

We use `sklearn.datasets.load_breast_cancer` — a well-known binary classification benchmark (569 samples, 30 features) that is challenging enough to show ensemble benefits but small enough to train quickly with shallow from-scratch trees.

In [2]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X_all, y_all = data.data.astype(np.float64), data.target

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.25, random_state=42, stratify=y_all
)

print(f"Train: {X_train.shape}  Test: {X_test.shape}")
print(f"Class balance — train: {y_train.mean():.3f}  test: {y_test.mean():.3f}")


Train: (426, 30)  Test: (143, 30)
Class balance — train: 0.627  test: 0.629


## Decision Tree from Scratch

Each tree in the forest is a shallow binary decision tree that:
- Splits on the feature/threshold pair that minimises weighted **Gini impurity**.
- Considers only a random subset of `max_features` features per split (key to decorrelation).
- Recurses until `max_depth` is reached or a node is pure.

Gini impurity for a node with class proportion p:
```
Gini(p) = 1 - p² - (1-p)²  =  2p(1-p)
```
Weighted impurity after splitting at threshold t:
```
Gini_split = (n_left/n) · Gini(left) + (n_right/n) · Gini(right)
```

In [3]:
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Optional


def _gini(y: np.ndarray) -> float:
    """Gini impurity for a binary label array."""
    if len(y) == 0:
        return 0.0
    p = y.mean()
    return 2.0 * p * (1.0 - p)


@dataclass
class _Node:
    """A node in a binary decision tree."""
    feature: Optional[int] = None
    threshold: Optional[float] = None
    left: Optional["_Node"] = None
    right: Optional["_Node"] = None
    prediction: Optional[float] = None  # leaf: majority class (int) or mean residual (float)


class ScratchDecisionTree:
    """Shallow binary classification tree with Gini splits and random feature subsets."""

    def __init__(self, max_depth: int = 4, max_features: Optional[int] = None, rng: Optional[np.random.Generator] = None) -> None:
        self.max_depth = max_depth
        self.max_features = max_features
        self.rng = rng or np.random.default_rng()
        self.root: Optional[_Node] = None

    def _best_split(self, X: np.ndarray, y: np.ndarray) -> tuple[Optional[int], Optional[float]]:
        n, d = X.shape
        # Random feature subset
        n_features = self.max_features or d
        feature_indices = self.rng.choice(d, size=min(n_features, d), replace=False)

        best_gini = float("inf")
        best_feat: Optional[int] = None
        best_thr: Optional[float] = None

        for feat in feature_indices:
            thresholds = np.unique(X[:, feat])
            for thr in thresholds[:-1]:  # skip last to ensure non-empty splits
                left_mask = X[:, feat] <= thr
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                g = (left_mask.sum() * _gini(y[left_mask]) + right_mask.sum() * _gini(y[right_mask])) / n
                if g < best_gini:
                    best_gini, best_feat, best_thr = g, feat, thr

        return best_feat, best_thr

    def _build(self, X: np.ndarray, y: np.ndarray, depth: int) -> _Node:
        # Leaf condition: max depth reached, pure node, or too few samples
        if depth >= self.max_depth or len(np.unique(y)) == 1 or len(y) < 2:
            return _Node(prediction=float(np.round(y.mean())))

        feat, thr = self._best_split(X, y)
        if feat is None:
            return _Node(prediction=float(np.round(y.mean())))

        left_mask = X[:, feat] <= thr
        return _Node(
            feature=feat,
            threshold=thr,
            left=self._build(X[left_mask], y[left_mask], depth + 1),
            right=self._build(X[~left_mask], y[~left_mask], depth + 1),
        )

    def fit(self, X: np.ndarray, y: np.ndarray) -> "ScratchDecisionTree":
        self.root = self._build(X, y, 0)
        return self

    def _predict_one(self, node: _Node, x: np.ndarray) -> int:
        if node.prediction is not None:
            return node.prediction
        if x[node.feature] <= node.threshold:  # type: ignore[index]
            return self._predict_one(node.left, x)  # type: ignore[arg-type]
        return self._predict_one(node.right, x)  # type: ignore[arg-type]

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.array([self._predict_one(self.root, x) for x in X])  # type: ignore[arg-type]


# Quick single-tree sanity check
rng0 = np.random.default_rng(0)
single_tree = ScratchDecisionTree(max_depth=4, rng=rng0)
single_tree.fit(X_train, y_train)
single_tree_acc = (single_tree.predict(X_test) == y_test).mean()
print(f"Single scratch tree test accuracy: {single_tree_acc:.3f}")

Single scratch tree test accuracy: 0.937


## Random Forest from Scratch

A Random Forest is an ensemble of B trees where:
- Each tree is trained on a **bootstrap sample** (n samples drawn with replacement).
- Each split considers a random subset of `max_features = sqrt(d)` features.
- Predictions are made by **majority vote** across all trees.

Bootstrapping + feature subsampling decorrelates trees so averaging reduces variance without increasing bias.

In [4]:
class ScratchRandomForest:
    """Random forest classifier from scratch."""

    def __init__(self, n_estimators: int = 50, max_depth: int = 6, max_features: Optional[int] = None, seed: int = 0) -> None:
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_features = max_features  # None → sqrt(d) set at fit time
        self.seed = seed
        self.trees: list[ScratchDecisionTree] = []

    def fit(self, X: np.ndarray, y: np.ndarray) -> "ScratchRandomForest":
        n, d = X.shape
        max_features = self.max_features or max(1, int(np.sqrt(d)))
        self.trees = []
        rng = np.random.default_rng(self.seed)

        for _ in range(self.n_estimators):
            # Bootstrap sample
            idx = rng.integers(0, n, size=n)
            X_boot, y_boot = X[idx], y[idx]
            tree = ScratchDecisionTree(max_depth=self.max_depth, max_features=max_features, rng=rng)
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)

        return self

    def predict(self, X: np.ndarray) -> np.ndarray:
        # Majority vote: shape (n_estimators, n_samples)
        votes = np.stack([t.predict(X) for t in self.trees], axis=0)
        return (votes.mean(axis=0) >= 0.5).astype(int)


print("Training scratch Random Forest (50 trees, max_depth=6)...")
scratch_rf = ScratchRandomForest(n_estimators=50, max_depth=6, seed=42)
scratch_rf.fit(X_train, y_train)
scratch_rf_acc = (scratch_rf.predict(X_test) == y_test).mean()
print(f"Scratch RF test accuracy: {scratch_rf_acc:.3f}")


Training scratch Random Forest (50 trees, max_depth=6)...


Scratch RF test accuracy: 0.958


## Gradient-Boosted Trees from Scratch

Gradient boosting builds an additive model **F_M(x) = Σ η · f_m(x)** where each `f_m` is a shallow **regression** tree fitted to the negative gradient of the loss with respect to the current predictions.

For **binary cross-entropy** loss with prediction probability p = sigmoid(F):
```
L = -[ y log p + (1-y) log(1-p) ]
dL/dF = p - y      (sigmoid output minus label)
```
So the pseudo-residual is `r_i = y_i - p_i`, and each tree is fitted to these residuals.

> This simplified implementation uses decision tree **regressors** (predict mean residual in each leaf) and a constant initial prediction.  It mirrors Friedman's Algorithm 1 for binary classification.

In [5]:
def _sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30, 30)))


class ScratchGBRegStump:
    """Shallow regression tree (stump/small tree) for gradient boosting.

    Predicts the mean of the target in each leaf.
    """

    def __init__(self, max_depth: int = 2, rng: Optional[np.random.Generator] = None) -> None:
        self.max_depth = max_depth
        self.rng = rng or np.random.default_rng()
        self.root: Optional[_Node] = None

    def _best_split_reg(self, X: np.ndarray, r: np.ndarray) -> tuple[Optional[int], Optional[float]]:
        """Best split by minimising sum of squared residuals."""
        n, d = X.shape
        best_sse = float("inf")
        best_feat: Optional[int] = None
        best_thr: Optional[float] = None

        for feat in range(d):
            thresholds = np.unique(X[:, feat])
            for thr in thresholds[:-1]:
                left_mask = X[:, feat] <= thr
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue
                sse = (np.var(r[left_mask]) * left_mask.sum() + np.var(r[right_mask]) * right_mask.sum())
                if sse < best_sse:
                    best_sse, best_feat, best_thr = sse, feat, thr

        return best_feat, best_thr

    def _build_reg(self, X: np.ndarray, r: np.ndarray, depth: int) -> _Node:
        if depth >= self.max_depth or len(r) < 2:
            return _Node(prediction=float(r.mean()))

        feat, thr = self._best_split_reg(X, r)
        if feat is None:
            return _Node(prediction=float(r.mean()))

        left_mask = X[:, feat] <= thr
        return _Node(
            feature=feat,
            threshold=thr,
            left=self._build_reg(X[left_mask], r[left_mask], depth + 1),
            right=self._build_reg(X[~left_mask], r[~left_mask], depth + 1),
        )

    def fit(self, X: np.ndarray, residuals: np.ndarray) -> "ScratchGBRegStump":
        self.root = self._build_reg(X, residuals, 0)
        return self

    def _pred_one(self, node: _Node, x: np.ndarray) -> float:
        if node.prediction is not None:
            return float(node.prediction)
        if x[node.feature] <= node.threshold:  # type: ignore[index]
            return self._pred_one(node.left, x)  # type: ignore[arg-type]
        return self._pred_one(node.right, x)  # type: ignore[arg-type]

    def predict(self, X: np.ndarray) -> np.ndarray:
        return np.array([self._pred_one(self.root, x) for x in X])  # type: ignore[arg-type]


class ScratchGradientBoosting:
    """Gradient-boosted classifier (binary cross-entropy loss) from scratch."""

    def __init__(self, n_estimators: int = 30, learning_rate: float = 0.1, max_depth: int = 2, seed: int = 0) -> None:
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.seed = seed
        self.trees: list[ScratchGBRegStump] = []
        self.F0: float = 0.0  # initial log-odds

    def fit(self, X: np.ndarray, y: np.ndarray) -> "ScratchGradientBoosting":
        rng = np.random.default_rng(self.seed)
        # Initial prediction: log-odds of mean class probability
        p_bar = y.mean().clip(1e-7, 1 - 1e-7)
        self.F0 = float(np.log(p_bar / (1.0 - p_bar)))

        F = np.full(len(y), self.F0, dtype=np.float64)
        self.trees = []

        for _ in range(self.n_estimators):
            p = _sigmoid(F)
            residuals = y.astype(np.float64) - p  # negative gradient of cross-entropy
            stump = ScratchGBRegStump(max_depth=self.max_depth, rng=rng)
            stump.fit(X, residuals)
            F = F + self.learning_rate * stump.predict(X)
            self.trees.append(stump)

        return self

    def decision_function(self, X: np.ndarray) -> np.ndarray:
        F = np.full(len(X), self.F0, dtype=np.float64)
        for tree in self.trees:
            F = F + self.learning_rate * tree.predict(X)
        return F

    def predict(self, X: np.ndarray) -> np.ndarray:
        return (_sigmoid(self.decision_function(X)) >= 0.5).astype(int)


print("Training scratch Gradient Boosting (30 stumps, max_depth=2)...")
scratch_gb = ScratchGradientBoosting(n_estimators=30, learning_rate=0.1, max_depth=2, seed=42)
scratch_gb.fit(X_train, y_train)
scratch_gb_acc = (scratch_gb.predict(X_test) == y_test).mean()
print(f"Scratch Gradient Boosting test accuracy: {scratch_gb_acc:.3f}")

# Single-tree baseline: one regression stump fitted directly to the raw labels (not
# iterated GBM rounds). This is a "one-tree" reference, not iteration-1 of the boosted
# ensemble, which would already be conditioned on F0.
single_tree_baseline = ScratchGBRegStump(max_depth=2)
single_tree_baseline.fit(X_train, y_train.astype(np.float64))
single_tree_baseline_preds = (single_tree_baseline.predict(X_test) >= 0.5).astype(int)
single_tree_baseline_acc = (single_tree_baseline_preds == y_test).mean()
print(f"Single-tree baseline accuracy (no boosting): {single_tree_baseline_acc:.3f}")
assert scratch_gb_acc > single_tree_baseline_acc, (
    f"Boosting should improve over single-tree baseline: {scratch_gb_acc:.3f} vs {single_tree_baseline_acc:.3f}"
)
print("Assertion passed: boosting improves over single-tree baseline.")

Training scratch Gradient Boosting (30 stumps, max_depth=2)...


Scratch Gradient Boosting test accuracy: 0.944


Single-tree baseline accuracy (no boosting): 0.916
Assertion passed: boosting improves over single-tree baseline.


## Validation Against scikit-learn

We compare our scratch implementations against `sklearn.ensemble.RandomForestClassifier` and `sklearn.ensemble.GradientBoostingClassifier`.

> **XGBoost note:** XGBoost is the production-grade gradient-boosting library used in practice. It adds regularisation on leaf weights and tree complexity (L1/L2), uses second-order loss approximations (Newton steps) for faster convergence, employs histogram-based split finding for speed and memory efficiency, and supports GPU acceleration and parallel training. In this notebook we validate against `sklearn.ensemble.GradientBoostingClassifier` instead, which shares the same core algorithm; `xgboost` is not installed in this environment.

In [6]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

# --- sklearn Random Forest ---
sk_rf = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=1)
sk_rf.fit(X_train, y_train)
sk_rf_acc = sk_rf.score(X_test, y_test)

# --- sklearn Gradient Boosting ---
sk_gb = GradientBoostingClassifier(n_estimators=30, learning_rate=0.1, max_depth=2, random_state=42)
sk_gb.fit(X_train, y_train)
sk_gb_acc = sk_gb.score(X_test, y_test)

# --- sklearn single decision tree (baseline) ---
sk_tree = DecisionTreeClassifier(max_depth=6, random_state=42)
sk_tree.fit(X_train, y_train)
sk_tree_acc = sk_tree.score(X_test, y_test)

print("=== Accuracy Comparison ===")
print(f"  Single decision tree (sklearn):          {sk_tree_acc:.3f}")
print(f"  Scratch RF (50 trees, depth 6):          {scratch_rf_acc:.3f}")
print(f"  sklearn RF (50 trees, depth 6):          {sk_rf_acc:.3f}")
print(f"  Scratch GB (30 stumps, lr=0.1, depth 2): {scratch_gb_acc:.3f}")
print(f"  sklearn GB (30 stumps, lr=0.1, depth 2): {sk_gb_acc:.3f}")

# --- Assertions ---
tolerance = 0.05
assert abs(scratch_rf_acc - sk_rf_acc) <= tolerance, (
    f"Scratch RF ({scratch_rf_acc:.3f}) too far from sklearn RF ({sk_rf_acc:.3f}), "
    f"diff={abs(scratch_rf_acc - sk_rf_acc):.3f} > {tolerance}"
)
print(f"\nAssertion passed: |scratch RF - sklearn RF| = "
      f"{abs(scratch_rf_acc - sk_rf_acc):.3f} <= {tolerance}")

assert abs(scratch_gb_acc - sk_gb_acc) <= tolerance, (
    f"scratch GB {scratch_gb_acc:.3f} vs sklearn GB {sk_gb_acc:.3f}, diff > {tolerance}"
)
print(f"Assertion passed: |scratch GB - sklearn GB| = "
      f"{abs(scratch_gb_acc - sk_gb_acc):.3f} <= {tolerance}")

=== Accuracy Comparison ===
  Single decision tree (sklearn):          0.937
  Scratch RF (50 trees, depth 6):          0.958
  sklearn RF (50 trees, depth 6):          0.951
  Scratch GB (30 stumps, lr=0.1, depth 2): 0.944
  sklearn GB (30 stumps, lr=0.1, depth 2): 0.951

Assertion passed: |scratch RF - sklearn RF| = 0.007 <= 0.05
Assertion passed: |scratch GB - sklearn GB| = 0.007 <= 0.05


## Visualisation

Left: Accuracy of our scratch implementations vs sklearn baselines.  Right: Cumulative accuracy of the scratch RF as we add more trees — shows how ensemble averaging steadily improves over a single tree.

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# --- Bar chart comparison ---
labels = ["Single tree\n(sklearn)", "Scratch RF\n(50 trees)", "sklearn RF\n(50 trees)",
          "Scratch GB\n(30 stumps)", "sklearn GB\n(30 stumps)"]
accs = [sk_tree_acc, scratch_rf_acc, sk_rf_acc, scratch_gb_acc, sk_gb_acc]
colors = ["#9ecae1", "#2171b5", "#6baed6", "#fd8d3c", "#e6550d"]
bars = axes[0].bar(labels, accs, color=colors, edgecolor="white", linewidth=0.8)
axes[0].set_ylim(0.85, 1.01)
axes[0].set_ylabel("Test Accuracy")
axes[0].set_title("Scratch vs sklearn — Breast Cancer")
for bar, acc in zip(bars, accs):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f"{acc:.3f}", ha="center", va="bottom", fontsize=9)

# --- Cumulative RF accuracy as trees are added ---
cum_accs = []
# Rebuild cumulative predictions incrementally
all_votes = np.stack([t.predict(X_test) for t in scratch_rf.trees], axis=0)  # (B, n)
for b in range(1, len(scratch_rf.trees) + 1):
    preds = (all_votes[:b].mean(axis=0) >= 0.5).astype(int)
    cum_accs.append((preds == y_test).mean())

axes[1].plot(range(1, len(cum_accs) + 1), cum_accs, color="#2171b5", lw=1.5, label="Scratch RF")
axes[1].axhline(single_tree_acc, color="grey", ls="--", lw=1, label="Single tree (scratch)")
axes[1].axhline(sk_rf_acc, color="#6baed6", ls=":", lw=1.5, label="sklearn RF (50 trees)")
axes[1].set_xlabel("Number of Trees")
axes[1].set_ylabel("Test Accuracy")
axes[1].set_title("Random Forest: Accuracy vs. Number of Trees")
axes[1].legend(fontsize=8)
axes[1].set_ylim(0.85, 1.01)

plt.tight_layout()
plt.savefig("rf_gb_plots.png", dpi=120)
plt.show()
print("Saved rf_gb_plots.png")


Saved rf_gb_plots.png


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_75035/196337527.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Takeaways

**Random Forests**
- Bootstrap sampling + random feature subsets **decorrelate** trees, making averaging effective at reducing variance without increasing bias.
- More trees → lower variance; accuracy plateaus around 50–100 trees for most datasets.
- Robust default: `max_features = sqrt(d)` for classification, `d/3` for regression.

**Gradient Boosting**
- Fits each new tree to the **negative gradient** of the loss, turning optimisation in function space into iterative residual fitting.
- For binary cross-entropy the residual is `y - sigmoid(F)` — the prediction error in probability space.
- Even 30 shallow stumps (depth=2) can match deep single trees because each stump contributes a small correction rather than a full prediction.

**XGBoost (production)**
XGBoost extends this algorithm with:
- L1/L2 regularisation on leaf weights and tree complexity.
- Second-order (Newton) loss approximations for faster convergence.
- Histogram-based split finding (speed + memory).
- Parallel and GPU training.

**Practical guidance**
- Start with Random Forest for a robust, low-maintenance baseline.
- Switch to XGBoost/LightGBM when you need the extra accuracy and can afford careful hyperparameter tuning with early stopping.
- Always validate with a held-out set; boosted models overfit easily without early stopping.
